<a href="https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagdyTarek18/Week1-Repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is content decline detection. I will treat this as a binary classification problem because I want to classify each content item as declining or not declining. The decision this supports is whether a content editor or SEO specialist should review a page for possible refresh or optimization. If a page is classified as declining, the editor can investigate it and decide whether it should be updated, rewritten, or improved.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("One row = one content item")

classification_columns = [
    "content_id",
    "client_id",
    "content_type",
    "ctr",
    "avg_position",
    "engagement_rate",
    "trend_direction"
]

display(df[classification_columns].head())

print("\nTrend direction distribution:")
print(df["trend_direction"].value_counts())

print("\nTrend direction percentages:")
print(
    (df["trend_direction"].value_counts(normalize=True) * 100).round(2)
)

Dataset shape: (30000, 44)
One row = one content item


,content_id,client_id,content_type,ctr,avg_position,engagement_rate,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,0.76,10.6,5.88,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,0.05,20.3,0.00,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.09,36.5,0.00,down
3,content_331d6c4de07b,client_19581e27de,keyword article,0.49,6.2,1.28,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.13,44.0,0.00,down



Trend direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend direction percentages:
trend_direction
down      54.21
stable    19.87
up        14.63
new        7.45
flat       3.84
Name: proportion, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is a binary variable called is_declining. I create it from trend_direction: content with trend_direction = "down" is labeled 1, and all other content is labeled 0. This lets me frame the problem as binary classification.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining"] = (
    df["trend_direction"] == "down"
).astype(int)


display(
    df[
        [
            "content_id",
            "trend_direction",
            "is_declining"
        ]
    ].head(10)
)


print("Target distribution:")
print(df["is_declining"].value_counts())

print("\nTarget percentages:")
print(
    (df["is_declining"].value_counts(normalize=True) * 100).round(2)
)

,content_id,trend_direction,is_declining
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64

Target percentages:
is_declining
1    54.21
0    45.79
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: F1-score

I will use F1-score as the main success metric because both types of classification mistakes matter in this problem. A false positive would cause an editor to spend time reviewing content that is not actually declining, while a false negative would cause the editor to miss content that may need attention. F1-score balances precision and recall, so it is more suitable than looking only at accuracy.

I will also compare the model against a simple baseline so that a model is only considered useful if it performs better than a naive prediction strategy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import f1_score


y = df["is_declining"]

print("Success metric: F1-score")

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentages:")
print(
    (y.value_counts(normalize=True) * 100).round(2)
)

def calculate_f1(y_true, y_pred):
    return f1_score(y_true, y_pred)

print("\nF1-score function is ready for when we have model predictions.")

Success metric: F1-score

Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64

Target percentages:
is_declining
1    54.21
0    45.79
Name: proportion, dtype: float64

F1-score function is ready for when we have model predictions.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis

One row represents one pseudonymized content item belonging to one pseudonymized client.

Each row contains information about the content item itself and its recent performance, such as impressions, clicks, CTR, average search position, engagement rate, and how recently the content was updated.

For this classification task, the target column is is_declining, where:

1 = declining
0 = not declining

content_id and client_id are identifiers used to distinguish content and clients. They are not intended to be predictive model features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

analysis_df = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "engagement_rate",
        "days_since_last_update",
        "is_declining"
    ]
].copy()

print("Unit of analysis:")
print("One row = one pseudonymized content item belonging to one client")

print("\nDataframe shape:")
print(analysis_df.shape)

print("\nFirst 10 rows:")
display(analysis_df.head(10))

Unit of analysis:
One row = one pseudonymized content item belonging to one client

Dataframe shape:
(30000, 10)

First 10 rows:


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,days_since_last_update,is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,5.88,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,0.00,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,0.00,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,1.28,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,0.00,14,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,0.03,8.5,0.00,20,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,0.00,7.0,0.00,20,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,0.06,21.2,3.57,22,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,0.09,46.0,5.88,20,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,0.16,4.9,0.00,104,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple fixed rule could be:

if avg_position > 20: predict declining

This rule is easy to understand, but it only considers one signal. Content decline can depend on multiple factors at the same time, such as CTR, engagement rate, impressions, clicks, content age, and days since the last update.

ML can combine several of these signals and learn more complex relationships that a single hand-written threshold cannot capture well.

However, ML should only be used if it performs better than a simple rule baseline. If the fixed rule performs just as well, the simpler rule would be preferable.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["rule_prediction"] = (df["avg_position"] > 20).astype(int)

# Compare the rule prediction with our target
from sklearn.metrics import f1_score

rule_f1 = f1_score(
    df["is_declining"],
    df["rule_prediction"]
)

print("Fixed rule:")
print("avg_position > 20 -> declining")

print("\nF1-score of fixed rule:")
print(round(rule_f1, 3))

print("\nPrediction distribution:")
print(df["rule_prediction"].value_counts())

Fixed rule:
avg_position > 20 -> declining

F1-score of fixed rule:
0.364

Prediction distribution:
rule_prediction
0    21461
1     8539
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.